In [ ]:
"""
Enhanced Transformer for route difficulty prediction with spatial awareness.
"""
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from transformers import BertConfig, BertModel, get_linear_schedule_with_warmup
from tqdm import tqdm
from collections import Counter
import datetime
import re

from src.data_processing import HOLD_ID, DataPreprocessing
from src.evaluation import Evaluation


class SpatialKilterBERT(nn.Module):
    """BERT with 2D positional embeddings and metadata integration."""
    
    def __init__(self, vocab_size, hidden_dim=128, num_layers=4, num_heads=8, dropout=0.1):
        super().__init__()
        
        config = BertConfig(
            vocab_size=vocab_size,
            hidden_size=hidden_dim,
            num_hidden_layers=num_layers,
            num_attention_heads=num_heads,
            intermediate_size=hidden_dim * 4,
            hidden_dropout_prob=dropout,
            attention_probs_dropout_prob=dropout,
            max_position_embeddings=100,
            pad_token_id=0
        )
        
        self.bert = BertModel(config)
        
        # 2D positional embeddings
        self.pos_x_embed = nn.Embedding(18, hidden_dim // 2)
        self.pos_y_embed = nn.Embedding(12, hidden_dim // 2)
        
        # Metadata projection
        self.angle_proj = nn.Linear(1, hidden_dim)
        self.metadata_proj = nn.Linear(2, hidden_dim)
        
        # Regression head
        self.regressor = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
    
    def forward(self, input_ids, pos_x, pos_y, angle, metadata, attention_mask=None):
        bert_out = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )
        
        pos_embed = torch.cat([
            self.pos_x_embed(pos_x),
            self.pos_y_embed(pos_y)
        ], dim=-1)
        
        spatial_hidden = bert_out.last_hidden_state + pos_embed
        cls_token = spatial_hidden[:, 0]
        
        angle_emb = self.angle_proj(angle)
        meta_emb = self.metadata_proj(metadata)
        combined = torch.cat([cls_token + angle_emb, meta_emb], dim=-1)
        
        return self.regressor(combined).squeeze(-1), bert_out.attentions


class EnhancedBoulderDataset(Dataset):
    """Dataset with spatial features. Works with both pandas DataFrames and HuggingFace Datasets."""
    
    def __init__(self, routes_data, vocab, hold_stats, max_length=25, augment=False):
        self.routes = routes_data
        self.vocab = vocab
        self.hold_stats = hold_stats
        self.max_length = max_length
        self.augment = augment
    
    def __len__(self):
        return len(self.routes)
    
    def __getitem__(self, idx):
        row = self.routes[idx]
        holds = row['holds_data']
        
        # Sort holds by position
        holds_sorted = sorted(holds, key=lambda h: (
            list(h.values())[0] not in [14, 15],
            -list(h.keys())[0] // 100,
            list(h.keys())[0] % 100
        ))
        
        # Augment: shuffle non-start/finish holds
        if self.augment and len(holds_sorted) > 2:
            start_finish = [h for h in holds_sorted if list(h.values())[0] in [14, 15]]
            middle = [h for h in holds_sorted if list(h.values())[0] not in [14, 15]]
            np.random.shuffle(middle)
            holds_sorted = start_finish + middle
        
        tokens, pos_x, pos_y = self._tokenize_with_positions(holds_sorted)
        
        # Metadata features
        density = len(holds) / self.max_length
        reaches = [abs(holds_sorted[i+1][list(holds_sorted[i+1].keys())[0]] - 
                      holds_sorted[i][list(holds_sorted[i].keys())[0]]) 
                  for i in range(len(holds_sorted)-1)]
        max_reach = max(reaches) / 1000 if reaches else 0
        
        angle = row['angle_y'] if row['angle_y'] is not None and str(row['angle_y']) != 'nan' else 0.0
        difficulty = row['display_difficulty']
        
        # Pad sequences
        attention_mask = [1] * len(tokens)
        pad_len = self.max_length - len(tokens)
        tokens.extend([0] * pad_len)
        pos_x.extend([0] * pad_len)
        pos_y.extend([0] * pad_len)
        attention_mask.extend([0] * pad_len)
        
        return {
            'input_ids': torch.LongTensor(tokens[:self.max_length]),
            'pos_x': torch.LongTensor(pos_x[:self.max_length]),
            'pos_y': torch.LongTensor(pos_y[:self.max_length]),
            'angle': torch.FloatTensor([angle]),
            'metadata': torch.FloatTensor([density, max_reach]),
            'attention_mask': torch.FloatTensor(attention_mask[:self.max_length]),
            'difficulty': torch.FloatTensor([difficulty]),
            'v_grade': row['v_grade']
        }
    
    def _tokenize_with_positions(self, holds):
        """Convert holds to tokens with 2D positions."""
        tokens = [1]  # [CLS]
        pos_x = [0]
        pos_y = [0]
        
        for hold in holds[:(self.max_length - 2)]:
            hold_id, func = list(hold.items())[0]
            
            hand_or_foot = 0 if func == 13 else 1
            token = self.vocab.get(f"{hold_id}_{hand_or_foot}", 0)
            tokens.append(token)
            
            x = hold_id % 100
            y = hold_id // 100
            pos_x.append(min(x, 17))
            pos_y.append(min(y, 11))
        
        tokens.append(2)  # [SEP]
        pos_x.append(0)
        pos_y.append(0)
        
        return tokens, pos_x, pos_y


class EnhancedKilterEncoder:
    """Enhanced encoder with hold statistics and training improvements."""
    
    def __init__(self, model_name='enhanced_transformer', hidden_dim=128, num_layers=4):
        self.model_name = model_name
        self.max_length = 25
        self.vocab = self._build_vocab()
        self.hold_stats = {}
        self.model = SpatialKilterBERT(len(self.vocab), hidden_dim, num_layers)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)
        self.scaler = GradScaler()
    
    def _build_vocab(self):
        """Build vocabulary with hand_or_foot encoding."""
        vocab = {'[PAD]': 0, '[CLS]': 1, '[SEP]': 2}
        token_id = 3
        for hold_id in HOLD_ID:
            for hand_or_foot in [0, 1]:
                vocab[f"{hold_id}_{hand_or_foot}"] = token_id
                token_id += 1
        return vocab
    
    def compute_hold_statistics(self, routes_data):
        """Compute hold frequency and co-occurrence by grade."""
        # Convert to list if needed for iteration
        routes_list = list(routes_data) if hasattr(routes_data, '__iter__') else [routes_data[i] for i in range(len(routes_data))]
        
        # Get unique grades
        unique_grades = set(row['v_grade'] for row in routes_list)
        hold_freq = {grade: Counter() for grade in unique_grades}
        hold_cooccur = Counter()
        
        for row in routes_list:
            grade = row['v_grade']
            holds = [list(h.items())[0] for h in row['holds_data']]
            
            for hold_id, func in holds:
                hand_or_foot = 0 if func == 13 else 1
                hold_freq[grade][f"{hold_id}_{hand_or_foot}"] += 1
            
            for i in range(len(holds)):
                for j in range(i+1, len(holds)):
                    hf1 = 0 if holds[i][1] == 13 else 1
                    hf2 = 0 if holds[j][1] == 13 else 1
                    pair = tuple(sorted([f"{holds[i][0]}_{hf1}", f"{holds[j][0]}_{hf2}"]))
                    hold_cooccur[pair] += 1
        
        self.hold_stats = {'frequency': hold_freq, 'cooccurrence': hold_cooccur}
        return self.hold_stats
    
    def train_model(self, train_data, val_data, epochs=30, batch_size=64, lr=2e-4, early_stop=3):
        """Train with mixed precision and early stopping."""
        self.compute_hold_statistics(train_data)
        
        train_dataset = EnhancedBoulderDataset(train_data, self.vocab, self.hold_stats, augment=True)
        val_dataset = EnhancedBoulderDataset(val_data, self.vocab, self.hold_stats, augment=False)
        
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size)
        
        optimizer = torch.optim.AdamW(self.model.parameters(), lr=lr, weight_decay=0.01)
        scheduler = get_linear_schedule_with_warmup(
            optimizer, 
            num_warmup_steps=len(train_loader) * epochs // 10,
            num_training_steps=len(train_loader) * epochs
        )
        
        grade_weights = self._compute_grade_weights(train_data)
        
        best_val_loss = float('inf')
        patience_counter = 0
        
        print(f"Training {self.model_name} ...")
        
        for epoch in range(epochs):
            train_loss = self._train_epoch(train_loader, optimizer, scheduler, grade_weights, epoch, epochs)
            val_loss = self._validate(val_loader, grade_weights)
            
            print(f"Epoch {epoch+1}/{epochs} | Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
                  f"LR: {scheduler.get_last_lr()[0]:.2e}")
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                self.save_model()
            else:
                patience_counter += 1
                if patience_counter >= early_stop:
                    print(f"Early stopping at epoch {epoch+1}")
                    break
            
            if epoch % 10 == 0 and epoch != 0:
                timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
                self.save_model(f"{self.model_name}_{epoch}_{timestamp}.pt")

    def _train_epoch(self, loader, optimizer, scheduler, grade_weights, epoch, total_epochs):
        """Single training epoch with mixed precision."""
        self.model.train()
        total_loss = 0
        
        for batch in tqdm(loader, desc=f"Epoch {epoch+1}/{total_epochs}", leave=False):
            v_grades = batch.pop('v_grade')
            batch = {k: v.to(self.device) for k, v in batch.items()}
            
            optimizer.zero_grad()
            
            with autocast():
                pred, _ = self.model(
                    batch['input_ids'], batch['pos_x'], batch['pos_y'],
                    batch['angle'], batch['metadata'], batch['attention_mask']
                )
                
                weights = torch.tensor([grade_weights.get(g, 1.0) for g in v_grades], 
                                      device=self.device)
                loss = (nn.SmoothL1Loss(reduction='none')(pred, batch['difficulty'].squeeze()) * weights).mean()
            
            self.scaler.scale(loss).backward()
            self.scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
            self.scaler.step(optimizer)
            self.scaler.update()
            scheduler.step()
            
            total_loss += loss.item()
        
        return total_loss / len(loader)
    
    def _validate(self, loader, grade_weights):
        """Validation with weighted loss."""
        self.model.eval()
        total_loss = 0
        
        with torch.no_grad():
            for batch in loader:
                v_grades = batch.pop('v_grade')
                batch = {k: v.to(self.device) for k, v in batch.items()}
                
                pred, _ = self.model(
                    batch['input_ids'], batch['pos_x'], batch['pos_y'],
                    batch['angle'], batch['metadata'], batch['attention_mask']
                )
                
                weights = torch.tensor([grade_weights.get(g, 1.0) for g in v_grades], 
                                      device=self.device)
                loss = (nn.SmoothL1Loss(reduction='none')(pred, batch['difficulty'].squeeze()) * weights).mean()
                total_loss += loss.item()
        
        return total_loss / len(loader)
    
    def _compute_grade_weights(self, data):
        """Compute weights to penalize errors on easier grades more."""
        # Convert to list for counting
        grades_list = [row['v_grade'] for row in data]
        grade_counts = Counter(grades_list)
        
        def grade_to_num(grade_str):
            match = re.search(r'\d+', str(grade_str))
            return int(match.group()) if match else 0
        
        max_grade_num = max(grade_to_num(g) for g in grade_counts.keys())
        
        weights = {grade: (max_grade_num - grade_to_num(grade) + 1) / max_grade_num 
                  for grade in grade_counts.keys()}
        return weights
    
    def predict(self, routes_data, batch_size=64):
        """Predict with attention extraction."""
        dataset = EnhancedBoulderDataset(routes_data, self.vocab, self.hold_stats, augment=False)
        loader = DataLoader(dataset, batch_size=batch_size)
        
        self.model.eval()
        predictions = []
        attentions = []
        
        with torch.no_grad():
            for batch in loader:
                batch.pop('v_grade')
                batch = {k: v.to(self.device) for k, v in batch.items()}
                
                pred, attn = self.model(
                    batch['input_ids'], batch['pos_x'], batch['pos_y'],
                    batch['angle'], batch['metadata'], batch['attention_mask']
                )
                
                predictions.extend(pred.cpu().numpy())
                if attn:
                    attentions.append(attn[-1].cpu().numpy())
        
        return np.array(predictions), attentions
    
    def save_model(self, name=None):
        if name is None:
            name = f'{self.model_name}.pt'
        torch.save({
            'model_state': self.model.state_dict(),
            'vocab': self.vocab,
            'hold_stats': self.hold_stats
        }, f'saved_models/{name}')
    
    def load_model(self, path=None):
        if path is None:
            path = f'{self.model_name}.pt'
        checkpoint = torch.load(f'saved_models/{path}', map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state'])
        self.vocab = checkpoint['vocab']
        self.hold_stats = checkpoint.get('hold_stats', {})
        self.model.eval()
    
    def model_summary(self):
        """Print model architecture and parameters."""
        print(f"\n{'='*60}")
        print(f"Model: {self.model_name}")
        print(f"{'='*60}")
        print(f"Vocabulary size: {len(self.vocab)}")
        print(f"Max sequence length: {self.max_length}")
        
        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        
        print(f"\nTotal parameters: {total_params:,}")
        print(f"Trainable parameters: {trainable_params:,}")
        print(f"Model size: {total_params * 4 / 1024**2:.2f} MB")
        
        print(f"\nArchitecture:")
        for name, module in self.model.named_children():
            params = sum(p.numel() for p in module.parameters())
            print(f"  {name}: {params:,} parameters")
        print(f"{'='*60}\n")